# Convergence — does more compute close the gap?

The Newton-vs-FEM gap has two different sources, one per solver family:

* **Newton (XPBD)** is an iterative positional projection — its *effective stiffness* depends on the number of solver iterations and substeps. We sweep both and watch the tip drop and the equilibrium-residual RMS approach the implicit-FEM / analytic value: XPBD converges towards true equilibrium, at a cost in iterations.
* **FEM (FEniCSx)** converges under mesh **h-refinement**. We watch the tip drop and strain energy reach a mesh-independent limit, compare it to the analytic 1-D bar (FEM **validity**), and confirm the *converged* answer is independent of the number of gravity load increments (a check on the nonlinear solve).

This separation — *solver* error (XPBD) vs *discretisation* error (FEM) — is the heart of the Newton-vs-FEM difference. Produce the data with `python -m newton_run.convergence` and `python -m fenics_run.convergence`.

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np

from common import params

nw = np.load(params.NEWTON_CONV_NPZ) if os.path.exists(params.NEWTON_CONV_NPZ) else None
fm = np.load(params.FEM_CONV_NPZ) if os.path.exists(params.FEM_CONV_NPZ) else None
print('Newton convergence:', nw is not None, '| FEM convergence:', fm is not None)

## 1. Newton (XPBD): tip drop & residual vs iterations / substeps

More iterations (and smaller substeps) tighten the positional projection. The
free-node equilibrium-residual RMS is the cleanest “have we converged” measure:
it should fall, and the tip drop should approach the FEM / analytic line.

In [ ]:
if nw is not None:
    tip_fem, tip_ana = float(nw['tip_fem']), float(nw['tip_analytic'])
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(nw['iters'], nw['it_tip'], 'o-', color='tab:orange', label='XPBD tip')
    if np.isfinite(tip_fem): ax[0].axhline(tip_fem, color='tab:blue', ls='--', label='FEM tet')
    ax[0].axhline(tip_ana, color='grey', ls=':', label='analytic 1-D')
    ax[0].set_xlabel(f"iterations (substeps={int(nw['fixed_substeps'])})"); ax[0].set_ylabel('tip drop [mm]')
    ax[0].set_title('tip vs iterations'); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].semilogy(nw['iters'], nw['it_res_rms'], 'o-', color='tab:red')
    ax[1].set_xlabel('iterations'); ax[1].set_ylabel('residual RMS [N]')
    ax[1].set_title('equilibrium residual vs iterations'); ax[1].grid(alpha=0.3)
    ax[2].plot(nw['substeps'], nw['sb_tip'], 's-', color='tab:orange', label='XPBD tip')
    if np.isfinite(tip_fem): ax[2].axhline(tip_fem, color='tab:blue', ls='--', label='FEM tet')
    ax[2].axhline(tip_ana, color='grey', ls=':', label='analytic 1-D')
    ax[2].set_xlabel(f"substeps (iters={int(nw['fixed_iters'])})"); ax[2].set_ylabel('tip drop [mm]')
    ax[2].set_title('tip vs substeps'); ax[2].legend(); ax[2].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"residual RMS: {nw['it_res_rms'][0]:.3g} N @ {int(nw['iters'][0])} iters -> "
          f"{nw['it_res_rms'][-1]:.3g} N @ {int(nw['iters'][-1])} iters")
else:
    print('No Newton convergence data. Run: python -m newton_run.convergence')

In [ ]:
if nw is not None:
    plt.figure(figsize=(6, 5))
    plt.loglog(nw['it_time'], nw['it_res_rms'], 'o-', color='tab:purple', label='vary iterations')
    plt.loglog(nw['sb_time'], nw['sb_res_rms'], 's-', color='tab:green', label='vary substeps')
    plt.xlabel('solve wall time [s]'); plt.ylabel('residual RMS [N]')
    plt.title('XPBD accuracy vs cost'); plt.legend(); plt.grid(alpha=0.3, which='both'); plt.show()

## 2. FEM: h-refinement convergence & validity vs analytic

The tip drop and strain energy converge to a mesh-independent limit as `h -> 0`.
That limit is the true 3-D answer; the analytic 1-D bar (dotted) is only an anchor
(it ignores Poisson + 3-D), so the gap that *remains* at the finest mesh is real
physics, not error. A FEM that converges cleanly and lands near the anchor is valid.

In [ ]:
if fm is not None:
    h = fm['h'] * 1000.0
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(h, fm['h_tip'], 'o-', color='tab:blue', label='FEM tip')
    ax[0].axhline(float(fm['tip_analytic']), color='grey', ls=':', label='analytic 1-D')
    ax[0].set_xlabel('mesh size h [mm]'); ax[0].set_ylabel('tip drop [mm]')
    ax[0].set_title('tip vs h'); ax[0].legend(); ax[0].grid(alpha=0.3); ax[0].invert_xaxis()
    ax[1].plot(h, fm['h_strain'], 'o-', color='tab:blue', label='FEM U')
    if 'strain_analytic' in fm.files:
        ax[1].axhline(float(fm['strain_analytic']), color='grey', ls=':', label='analytic 1-D')
    ax[1].set_xlabel('mesh size h [mm]'); ax[1].set_ylabel('strain energy [J]')
    ax[1].set_title('strain energy vs h'); ax[1].legend(); ax[1].grid(alpha=0.3); ax[1].invert_xaxis()
    ax[2].semilogx(fm['ndofs'], fm['h_tip'], 'o-', color='tab:blue')
    ax[2].set_xlabel('# DOFs'); ax[2].set_ylabel('tip drop [mm]')
    ax[2].set_title('tip vs problem size (cost)'); ax[2].grid(alpha=0.3, which='both')
    plt.tight_layout(); plt.show()
    print(f"finest tip = {fm['h_tip'][-1]:.3f} mm vs analytic 1-D {float(fm['tip_analytic']):.3f} mm")
else:
    print('No FEM convergence data. Run: python -m fenics_run.convergence')

## 3. FEM load-increment sweep — the converged tip must be flat

The number of gravity load increments is only a continuation strategy for the
Newton–Raphson solve; it must **not** change the converged answer. A flat tip-drop
curve (while the total Newton iteration count varies) validates the nonlinear solve.

In [ ]:
if fm is not None:
    fig, ax1 = plt.subplots(figsize=(6, 4))
    ax1.plot(fm['load_steps'], fm['ls_tip'], 'o-', color='tab:blue')
    ax1.set_xlabel('# gravity load increments'); ax1.set_ylabel('tip drop [mm]', color='tab:blue')
    ax2 = ax1.twinx()
    ax2.plot(fm['load_steps'], fm['ls_its'], 's--', color='tab:red')
    ax2.set_ylabel('total Newton iterations', color='tab:red')
    plt.title('FEM load-increment sweep'); plt.tight_layout(); plt.show()
    print(f"tip spread across load-step counts = {fm['ls_tip'].max() - fm['ls_tip'].min():.2e} mm "
          f"(should be ~0 -> nonlinear solve is correct)")

## Takeaway

* XPBD's answer is **solver-budget dependent**: too few iterations/substeps leave a
  large equilibrium residual (it under-resolves the elastic balance). It approaches
  the FEM/analytic value only as the budget grows — the price of a fast positional solver.
* FEM's answer is **discretisation dependent** but converges monotonically under
  h-refinement to a budget-independent equilibrium, and is independent of the load-step
  schedule. That separation — *solver* error vs *discretisation* error — is the essence
  of the Newton-vs-FEM difference.